In [8]:
# ============================================================
# MODULE 1 — IMPORTS AND SETUP ONLY
# ============================================================

import subprocess, sys
for pkg in ["pandas", "numpy", "plotly"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os
import base64
from datetime import datetime
from IPython.display import IFrame, HTML, display

print("MODULE 1 COMPLETE — pandas, numpy, plotly, IPython loaded.")

MODULE 1 COMPLETE — pandas, numpy, plotly, IPython loaded.


In [9]:
# ============================================================
# MODULE 2 — FUND REGISTRY (HARDCODED, SIX PROVIDERS)
# ============================================================
# All figures hardcoded from KIIDs, factsheets, and product pages
# as of October 2026. Source and date stated in the dashboard
# disclaimer. No fetching, no parsing. Ranks start at 1.

FUND_REGISTRY = [
    {
        "provider": "BlackRock",
        "fund_name": "iShares Core S&P 500 UCITS ETF",
        "isin": "IE00B5BMR087",
        "share_class": "USD Acc",
        "hedged": False,
        "listing": "LSE",
        "ter_pct": 0.07,
        "aum_musd": 154477.00,
        "holdings": 504,
        "risk": 6,
        "replication": "Physical",
        "distribution": "Accumulating",
        "inception": "18-May-2010",
    },
    {
        "provider": "State Street",
        "fund_name": "SPDR S&P 500 UCITS ETF (Acc)",
        "isin": "IE000XZSV718",
        "share_class": "USD Acc",
        "hedged": False,
        "listing": "LSE",
        "ter_pct": 0.03,
        "aum_musd": 19178.77,
        "holdings": 503,
        "risk": 6,
        "replication": "Physical",
        "distribution": "Accumulating",
        "inception": "31-Oct-2023",
    },
    {
        "provider": "Vanguard",
        "fund_name": "Vanguard S&P 500 UCITS ETF",
        "isin": "IE00BFMXXD54",
        "share_class": "USD Acc",
        "hedged": False,
        "listing": "LSE",
        "ter_pct": 0.07,
        "aum_musd": 36300.00,
        "holdings": 504,
        "risk": 6,
        "replication": "Physical",
        "distribution": "Accumulating",
        "inception": "14-May-2019",
    },
    {
        "provider": "Xtrackers",
        "fund_name": "Xtrackers S&P 500 UCITS ETF 4C",
        "isin": "IE000Z9SJA06",
        "share_class": "USD Acc",
        "hedged": False,
        "listing": "LSE",
        "ter_pct": 0.03,
        "aum_musd": 4210.00,
        "holdings": 504,
        "risk": 6,
        "replication": "Physical",
        "distribution": "Accumulating",
        "inception": "27-Feb-2015",
    },
    {
        "provider": "Amundi",
        "fund_name": "Amundi Core S&P 500 Swap UCITS ETF USD Dist",
        "isin": "LU0496786657",
        "share_class": "USD Dist",
        "hedged": False,
        "listing": "LSE",
        "ter_pct": 0.05,
        "aum_musd": 33571.34,
        "holdings": 503,
        "risk": 6,
        "replication": "Synthetic",
        "distribution": "Distributing",
        "inception": "26-Mar-2010",
    },
    {
        "provider": "Invesco",
        "fund_name": "Invesco S&P 500 UCITS ETF Acc",
        "isin": "IE00B3YCGJ38",
        "share_class": "USD Acc",
        "hedged": False,
        "listing": "LSE",
        "ter_pct": 0.05,
        "aum_musd": 59140.00,
        "holdings": 503,
        "risk": 6,
        "replication": "Synthetic",
        "distribution": "Accumulating",
        "inception": "20-May-2010",
    },
]

registry_df = pd.DataFrame(FUND_REGISTRY)
registry_df.index = range(1, len(registry_df) + 1)
registry_df.index.name = "fund_no"

# TER in basis points for charts and rankings
registry_df["ter_bps"] = registry_df["ter_pct"] * 100

print("=" * 78)
print("MODULE 2 COMPLETE — six-fund hardcoded registry loaded.")
print("=" * 78)
display(registry_df[[
    "provider", "fund_name", "isin", "share_class",
    "ter_pct", "ter_bps", "aum_musd", "holdings", "risk",
    "replication", "distribution", "inception",
]])

MODULE 2 COMPLETE — six-fund hardcoded registry loaded.


,provider,fund_name,isin,share_class,ter_pct,ter_bps,aum_musd,holdings,risk,replication,distribution,inception
fund_no,,,,,,,,,,,,
1,BlackRock,iShares Core S&P 500 UCITS ETF,IE00B5BMR087,USD Acc,0.07,7.0,154477.00,504,6,Physical,Accumulating,18-May-2010
2,State Street,SPDR S&P 500 UCITS ETF (Acc),IE000XZSV718,USD Acc,0.03,3.0,19178.77,503,6,Physical,Accumulating,31-Oct-2023
3,Vanguard,Vanguard S&P 500 UCITS ETF,IE00BFMXXD54,USD Acc,0.07,7.0,36300.00,504,6,Physical,Accumulating,14-May-2019
4,Xtrackers,Xtrackers S&P 500 UCITS ETF 4C,IE000Z9SJA06,USD Acc,0.03,3.0,4210.00,504,6,Physical,Accumulating,27-Feb-2015
5,Amundi,Amundi Core S&P 500 Swap UCITS ETF USD Dist,LU0496786657,USD Dist,0.05,5.0,33571.34,503,6,Synthetic,Distributing,26-Mar-2010
6,Invesco,Invesco S&P 500 UCITS ETF Acc,IE00B3YCGJ38,USD Acc,0.05,5.0,59140.00,503,6,Synthetic,Accumulating,20-May-2010


In [10]:
# ============================================================
# MODULE 3 — DATA ASSEMBLY AND VALIDATION
# ============================================================
# Build comparison_df from the hardcoded registry.
# Validate every field against sanity bounds.
# Compute 1-based ranks: cheapest = 1, largest = 1.
# No fetching, no parsing.

comparison_df = registry_df[[
    "provider", "fund_name", "isin", "share_class", "listing",
    "ter_pct", "ter_bps", "aum_musd", "holdings", "risk",
    "replication", "distribution", "inception",
]].copy()

# ---------- VALIDATION ----------

def _validate(row):
    issues = []
    if not (0.01 <= row["ter_pct"] <= 1.00):
        issues.append(f"TER {row['ter_pct']} outside 0.01–1.00")
    if row["aum_musd"] <= 10:
        issues.append(f"AUM {row['aum_musd']} below 10m floor")
    if not (1 <= row["risk"] <= 7):
        issues.append(f"Risk {row['risk']} outside 1–7")
    if row["holdings"] <= 0:
        issues.append(f"Holdings {row['holdings']} not positive")
    return issues

print("=" * 78)
print("MODULE 3 — VALIDATION")
print("=" * 78)

validation_clean = True
for _, row in comparison_df.iterrows():
    issues = _validate(row)
    if issues:
        validation_clean = False
        for issue in issues:
            print(f"   WARNING [{row['provider']}]: {issue}")

if validation_clean:
    print("   All six funds pass sanity bounds (TER 0.01–1.00, AUM >10m, risk 1–7, holdings >0).")

# ---------- RANKS (1-BASED) ----------

# ter_rank: cheapest = 1. Ties share the same rank (method="min").
comparison_df["ter_rank"] = (
    comparison_df["ter_bps"].rank(method="min", ascending=True).astype(int)
)

# aum_rank: largest = 1.
comparison_df["aum_rank"] = (
    comparison_df["aum_musd"].rank(method="min", ascending=False).astype(int)
)

# holdings_rank: fewest = 1 (informational only; holdings are not a quality signal).
comparison_df["holdings_rank"] = (
    comparison_df["holdings"].rank(method="min", ascending=True).astype(int)
)

# ---------- DERIVED COLUMNS ----------

comparison_df["aum_bn"] = (comparison_df["aum_musd"] / 1000).round(1)
comparison_df["ter_vs_cheapest_bps"] = comparison_df["ter_bps"] - comparison_df["ter_bps"].min()
comparison_df["aum_vs_largest_pct"] = (
    comparison_df["aum_musd"] / comparison_df["aum_musd"].max() * 100
).round(1)

# ---------- DATA STATUS ----------

comparison_df["data_status"] = "hardcoded — source cited in dashboard"

print("\n" + "=" * 78)
print("MODULE 3 COMPLETE — comparison_df built, validated, ranked (1-based).")
print("=" * 78)
display(comparison_df[[
    "provider", "ter_bps", "ter_rank", "aum_bn", "aum_rank",
    "ter_vs_cheapest_bps", "holdings", "holdings_rank", "risk",
    "replication", "distribution", "data_status",
]])

MODULE 3 — VALIDATION
   All six funds pass sanity bounds (TER 0.01–1.00, AUM >10m, risk 1–7, holdings >0).

MODULE 3 COMPLETE — comparison_df built, validated, ranked (1-based).


,provider,ter_bps,ter_rank,aum_bn,aum_rank,ter_vs_cheapest_bps,holdings,holdings_rank,risk,replication,distribution,data_status
fund_no,,,,,,,,,,,,
1,BlackRock,7.0,5,154.5,1,4.0,504,4,6,Physical,Accumulating,hardcoded — source cited in dashboard
2,State Street,3.0,1,19.2,5,0.0,503,1,6,Physical,Accumulating,hardcoded — source cited in dashboard
3,Vanguard,7.0,5,36.3,3,4.0,504,4,6,Physical,Accumulating,hardcoded — source cited in dashboard
4,Xtrackers,3.0,1,4.2,6,0.0,504,4,6,Physical,Accumulating,hardcoded — source cited in dashboard
5,Amundi,5.0,3,33.6,4,2.0,503,1,6,Synthetic,Distributing,hardcoded — source cited in dashboard
6,Invesco,5.0,3,59.1,2,2.0,503,1,6,Synthetic,Accumulating,hardcoded — source cited in dashboard


In [19]:
# ============================================================
# MODULE 4 — ANALYTICS
# ============================================================
# Cost comparison, scale comparison, composite efficiency labels,
# and the narrative the dashboard renders.
# All ranks 1-based. No em dashes or unnecessary hyphens in the narrative.

# ---------- 4.1 COST COMPARISON ----------

cheapest = comparison_df.loc[comparison_df["ter_bps"].idxmin()]
most_expensive = comparison_df.loc[comparison_df["ter_bps"].idxmax()]
spread_bps = int(most_expensive["ter_bps"] - cheapest["ter_bps"])

holding = 100_000
annual_cost_diff = holding * spread_bps / 10_000
twenty_year_simple = annual_cost_diff * 20

print("=" * 78)
print("4.1 COST COMPARISON")
print("=" * 78)
print(f"   Cheapest:        {cheapest['provider']} at {int(cheapest['ter_bps'])} bps")
print(f"   Most expensive:  {most_expensive['provider']} at {int(most_expensive['ter_bps'])} bps")
print(f"   Spread:          {spread_bps} bps")
print(f"   On 100,000:      {annual_cost_diff:,.0f} per year, "
      f"about {twenty_year_simple:,.0f} over twenty years before compounding")

# ---------- 4.2 SCALE COMPARISON ----------

largest = comparison_df.loc[comparison_df["aum_musd"].idxmax()]
smallest = comparison_df.loc[comparison_df["aum_musd"].idxmin()]
scale_multiple = largest["aum_musd"] / smallest["aum_musd"]

print("\n" + "=" * 78)
print("4.2 SCALE COMPARISON")
print("=" * 78)
print(f"   Largest:         {largest['provider']} at ${largest['aum_bn']}bn")
print(f"   Smallest:        {smallest['provider']} at ${smallest['aum_bn']}bn")
print(f"   Multiple:        {scale_multiple:.1f}x")
print(f"   Cheapest = largest? "
      f"{'Yes' if cheapest['provider'] == largest['provider'] else 'No'}")
print(f"   Cost and scale sit in "
      f"{'the same provider' if cheapest['provider'] == largest['provider'] else 'different providers'}.")

# ---------- 4.3 COMPOSITE EFFICIENCY LABELS ----------

cheapest_ter = comparison_df["ter_bps"].min()
labels = []

for _, row in comparison_df.iterrows():
    fund_labels = []
    if row["ter_bps"] == cheapest_ter:
        fund_labels.append("Cost leader")
    if row["provider"] == largest["provider"]:
        fund_labels.append("Scale leader")
    if (row["ter_bps"] <= cheapest_ter + 2) and (row["aum_rank"] <= 3):
        fund_labels.append("Balanced")
    if not fund_labels:
        fund_labels.append("None")
    labels.append(" | ".join(fund_labels))

comparison_df["efficiency_label"] = labels

print("\n" + "=" * 78)
print("4.3 COMPOSITE EFFICIENCY LABELS")
print("=" * 78)
for _, row in comparison_df.iterrows():
    print(f"   {row['provider']:<14} {row['efficiency_label']}")

# ---------- 4.4 NARRATIVE ----------

balanced_funds = comparison_df[
    comparison_df["efficiency_label"].str.contains("Balanced")
]
balanced_text = (
    ", ".join(balanced_funds["provider"].tolist())
    if len(balanced_funds) else "None"
)

synthetic_count = (comparison_df["replication"] == "Synthetic").sum()
physical_count = (comparison_df["replication"] == "Physical").sum()

narrative_headline = "S&P 500 UCITS ETF cost and scale comparison"

narrative_facts = [
    ("Cheapest provider", f"{cheapest['provider']} at {int(cheapest['ter_bps'])} bps"),
    ("Most expensive provider", f"{most_expensive['provider']} at {int(most_expensive['ter_bps'])} bps"),
    ("Spread across the set", f"{spread_bps} bps"),
    ("Largest fund", f"{largest['provider']} at ${largest['aum_bn']}bn"),
    ("Smallest fund", f"{smallest['provider']} at ${smallest['aum_bn']}bn"),
    ("Balanced funds", balanced_text),
]

narrative_body = (
    f"The cost spread of {spread_bps} bps between {cheapest['provider']} and "
    f"{most_expensive['provider']} looks small in percentage terms but compounds "
    f"into roughly {annual_cost_diff:,.0f} per year on a 100,000 position, or "
    f"about {twenty_year_simple:,.0f} over twenty years before compounding. "
    f"The cheapest funds, {cheapest['provider']} and Xtrackers at 3 bps, are not "
    f"the largest. {largest['provider']} dominates on scale at "
    f"${largest['aum_bn']}bn while charging {int(largest['ter_bps'])} bps. "
    f"That fragmentation is the central finding. Cost leadership and scale "
    f"leadership sit in different providers, so a client choosing purely on price "
    f"gives up liquidity and secondary market depth, and a client choosing purely "
    f"on scale pays for it. The set splits {physical_count} physical and "
    f"{synthetic_count} synthetic, yet the cheapest funds here are physical, so "
    f"the assumption that synthetic always costs less does not hold across this group."
)

print("\n" + "=" * 78)
print("4.4 NARRATIVE")
print("=" * 78)
print(narrative_headline)
for label, value in narrative_facts:
    print(f"   {label}: {value}")
print("\n" + narrative_body)

print("\n" + "=" * 78)
print("MODULE 4 COMPLETE — analytics and narrative built.")
print("=" * 78)

4.1 COST COMPARISON
   Cheapest:        State Street at 3 bps
   Most expensive:  BlackRock at 7 bps
   Spread:          4 bps
   On 100,000:      40 per year, about 800 over twenty years before compounding

4.2 SCALE COMPARISON
   Largest:         BlackRock at $154.5bn
   Smallest:        Xtrackers at $4.2bn
   Multiple:        36.7x
   Cheapest = largest? No
   Cost and scale sit in different providers.

4.3 COMPOSITE EFFICIENCY LABELS
   BlackRock      Scale leader
   State Street   Cost leader
   Vanguard       None
   Xtrackers      Cost leader
   Amundi         None
   Invesco        Balanced

4.4 NARRATIVE
S&P 500 UCITS ETF cost and scale comparison
   Cheapest provider: State Street at 3 bps
   Most expensive provider: BlackRock at 7 bps
   Spread across the set: 4 bps
   Largest fund: BlackRock at $154.5bn
   Smallest fund: Xtrackers at $4.2bn
   Balanced funds: Invesco

The cost spread of 4 bps between State Street and BlackRock looks small in percentage terms but compounds i

In [20]:
# ============================================================
# MODULE 5 — VISUALISATION AND HTML DASHBOARD
# ============================================================
# Builds etf_dashboard.html with header, disclaimer banner,
# KPI tiles, three charts, comparison table with source links,
# narrative card, and an in-page open-in-new-tab button.

# ---------- SOURCE URLs ----------
SOURCE_URLS = {
    "BlackRock": {
        "webpage_url": "https://www.blackrock.com/uk/individual/products/253743/ishares-sp-500-b-ucits-etf-acc-fund",
        "kiid_url": "https://www.blackrock.com/uk/literature/kiid/ucits_kiid-ishares-core-sp-500-ucits-etf-usd-acc-gb-ie00b5bmr087-en.pdf",
        "factsheet_url": "https://www.blackrock.com/uk/literature/fact-sheet/cspx-ishares-core-s-p-500-ucits-etf-fund-fact-sheet-en-gb.pdf",
    },
    "State Street": {
        "webpage_url": "https://www.ssga.com/uk/en_gb/intermediary/etfs/state-street-spdr-sp-500-ucits-etf-acc-spyl-gy",
        "kiid_url": "https://www.ssga.com/library-content/kiids?isin=IE000XZSV718&documentType=kiid&country=uk&language=en_gb&ticker=spyl-gy",
        "factsheet_url": "https://www.ssga.com/library-content/products/factsheets/etfs/emea/factsheet-emea-en_gb-spyl-gy.pdf",
    },
    "Vanguard": {
        "webpage_url": "https://www.vanguardinvestor.co.uk/investments/vanguard-s-and-p-500-ucits-etf-usd-accumulating/overview",
        "kiid_url": "https://fund-docs.vanguard.com/ie00bfmxxd54-en.pdf",
        "factsheet_url": None,
    },
    "Xtrackers": {
        "webpage_url": "https://etf.dws.com/en-gb/IE000Z9SJA06-s-p-500-ucits-etf-4c/",
        "kiid_url": "https://etf.dws.com/download/asset/838eee43-e5fb-422e-9d10-e028b5fe3494",
        "factsheet_url": "https://etf.dws.com/download/asset/8c8f9cdc-9c64-4314-a25b-8222c6344e90",
    },
    "Amundi": {
        "webpage_url": "https://www.amundietf.co.uk/en/individual/products/equity/amundi-core-sp-500-swap-ucits-etf-usd-dist/lu0496786657",
        "kiid_url": "https://www.amundietf.co.uk/pdfDocuments/kiid/LU0496786657/ENG/GBR/20260211",
        "factsheet_url": "https://www.amundietf.co.uk/pdfDocuments/monthly-factsheet/LU0496786657/ENG/GBR/RETAIL/ETF/20260930",
    },
    "Invesco": {
        "webpage_url": "https://www.invesco.com/uk/en/financial-products/etfs/invesco-sp-500-ucits-etf-acc.html#Holdings",
        "kiid_url": "https://www.invesco.com/content/dam/invesco/uk/en/product-documents/etf/share-class/kiid/IE00B3YCGJ38_kiid_en-uk.pdf",
        "factsheet_url": "https://www.invesco.com/content/dam/invesco/uk/en/product-documents/etf/share-class/factsheet/IE00B3YCGJ38_factsheet_en-uk.pdf",
    },
}

DISCLAIMER_DASHBOARD = (
    "This dashboard is an analytical exercise, not investment advice. All data is "
    "taken from provider published documents and product pages as of October 2026. "
    "The comparison uses unhedged accumulating share classes, except Amundi, which "
    "is distributing. Figures may differ from live market data and should be "
    "verified against the source links listed per fund in the comparison table."
)

generated_at = datetime.now().strftime("%d %B %Y, %H:%M")

NAVY = "#0b1f3a"
ACCENT = "#1f6feb"
PHYSICAL = "#1f6feb"
SYNTHETIC = "#f59e0b"
GREY = "#6b7280"

# ---------- CHART 1: TER BAR ----------
ter_sorted = comparison_df.sort_values("ter_bps", ascending=True)
ter_colors = [PHYSICAL if r == "Physical" else SYNTHETIC for r in ter_sorted["replication"]]

fig_ter = go.Figure(go.Bar(
    x=ter_sorted["ter_bps"], y=ter_sorted["provider"], orientation="h",
    marker_color=ter_colors,
    text=[f"{int(v)} bps" for v in ter_sorted["ter_bps"]],
    textposition="outside",
    hovertemplate="%{y}<br>TER: %{x} bps<extra></extra>",
))
fig_ter.update_layout(
    title="TER by provider (cheapest first)", xaxis_title="TER (basis points)", yaxis_title="",
    height=340, margin=dict(l=10, r=40, t=50, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="system-ui, -apple-system, Segoe UI, Roboto, sans-serif", color=NAVY),
    showlegend=False,
)

# ---------- CHART 2: AUM BAR ----------
aum_sorted = comparison_df.sort_values("aum_musd", ascending=True)
aum_colors = [PHYSICAL if r == "Physical" else SYNTHETIC for r in aum_sorted["replication"]]

fig_aum = go.Figure(go.Bar(
    x=aum_sorted["aum_bn"], y=aum_sorted["provider"], orientation="h",
    marker_color=aum_colors,
    text=[f"${v:.1f}bn" for v in aum_sorted["aum_bn"]],
    textposition="outside",
    hovertemplate="%{y}<br>AUM: $%{x:.1f}bn<extra></extra>",
))
fig_aum.update_layout(
    title="Fund size by provider (largest first)", xaxis_title="AUM (USD billions)", yaxis_title="",
    height=340, margin=dict(l=10, r=60, t=50, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="system-ui, -apple-system, Segoe UI, Roboto, sans-serif", color=NAVY),
    showlegend=False,
)

# ---------- CHART 3: TER vs AUM SCATTER ----------
scatter_colors = [PHYSICAL if r == "Physical" else SYNTHETIC for r in comparison_df["replication"]]

fig_scatter = go.Figure(go.Scatter(
    x=comparison_df["ter_bps"], y=comparison_df["aum_bn"],
    mode="markers+text", text=comparison_df["provider"], textposition="top center",
    marker=dict(size=16, color=scatter_colors, line=dict(width=1, color="white")),
    hovertemplate="%{text}<br>TER: %{x} bps<br>AUM: $%{y:.1f}bn<extra></extra>",
))
fig_scatter.update_layout(
    title="TER vs fund size", xaxis_title="TER (basis points)", yaxis_title="AUM (USD billions)",
    height=420, margin=dict(l=10, r=40, t=50, b=40),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="system-ui, -apple-system, Segoe UI, Roboto, sans-serif", color=NAVY),
    showlegend=False,
)

chart_ter_html = fig_ter.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})
chart_aum_html = fig_aum.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})
chart_scatter_html = fig_scatter.to_html(full_html=False, include_plotlyjs=False, config={"displayModeBar": False})

# ---------- KPI TILES ----------
kpi_cheapest = comparison_df.loc[comparison_df["ter_bps"].idxmin()]
kpi_largest = comparison_df.loc[comparison_df["aum_musd"].idxmax()]

kpi_html = f"""
<div class="kpi-row">
  <div class="kpi"><div class="kpi-value">{len(comparison_df)}</div>
       <div class="kpi-label">Funds compared</div></div>
  <div class="kpi"><div class="kpi-value">{int(kpi_cheapest['ter_bps'])} bps</div>
       <div class="kpi-label">Cheapest TER · {kpi_cheapest['provider']}</div></div>
  <div class="kpi"><div class="kpi-value">${kpi_largest['aum_bn']}bn</div>
       <div class="kpi-label">Largest fund · {kpi_largest['provider']}</div></div>
  <div class="kpi"><div class="kpi-value">{comparison_df['risk'].max()}/7</div>
       <div class="kpi-label">Risk indicator (all funds)</div></div>
</div>
"""

# ---------- SOURCE LINKS ----------
def _src_links(provider):
    u = SOURCE_URLS.get(provider, {})
    parts = []
    if u.get("webpage_url"):   parts.append(f'<a href="{u["webpage_url"]}" target="_blank">webpage</a>')
    if u.get("kiid_url"):      parts.append(f'<a href="{u["kiid_url"]}" target="_blank">KIID</a>')
    if u.get("factsheet_url"): parts.append(f'<a href="{u["factsheet_url"]}" target="_blank">factsheet</a>')
    return " · ".join(parts) if parts else "None"

# ---------- COMPARISON TABLE ----------
table_rows = []
for _, r in comparison_df.sort_values("ter_rank").iterrows():
    table_rows.append(f"""
    <tr>
      <td>{r['provider']}</td>
      <td>{r['fund_name']}</td>
      <td>{r['share_class']}</td>
      <td>{r['listing']}</td>
      <td class="num">{int(r['ter_bps'])}</td>
      <td class="num">${r['aum_bn']:.1f}bn</td>
      <td class="num">{r['holdings']}</td>
      <td class="num">{r['risk']}/7</td>
      <td>{r['replication']}</td>
      <td>{r['efficiency_label']}</td>
      <td>{_src_links(r['provider'])}</td>
    </tr>""")

table_html = f"""
<table class="compare">
  <thead>
    <tr>
      <th>Provider</th><th>Fund</th><th>Share class</th><th>Listing</th>
      <th class="num">TER (bps)</th><th class="num">AUM</th>
      <th class="num">Holdings</th><th class="num">Risk</th>
      <th>Replication</th><th>Label</th><th>Source</th>
    </tr>
  </thead>
  <tbody>{''.join(table_rows)}</tbody>
</table>
"""

# ---------- NARRATIVE CARD ----------
facts_html = "".join(
    f'<div class="fact"><div class="fact-label">{label}</div>'
    f'<div class="fact-value">{value}</div></div>'
    for label, value in narrative_facts
)

narrative_card = f"""
<div class="card">
  <h2>Narrative</h2>
  <p class="narrative-headline">{narrative_headline}</p>
  <div class="fact-grid">{facts_html}</div>
  <p class="narrative-body">{narrative_body}</p>
</div>
"""

# ---------- FULL HTML ----------
dashboard_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>ETF Cost Intelligence Engine</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
  body {{ margin:0; background:#f5f7fa; font-family: system-ui,-apple-system,"Segoe UI",Roboto,sans-serif; color:{NAVY}; }}
  .wrap {{ max-width:1100px; margin:0 auto; padding:24px; }}
  header {{ background:{NAVY}; color:white; padding:28px 32px; border-radius:14px; }}
  header h1 {{ margin:0; font-size:26px; }}
  header p {{ margin:6px 0 0; opacity:0.85; }}
  .generated {{ font-size:12px; opacity:0.65; margin-top:8px; }}
  .banner {{ background:#fff7ed; border-left:6px solid #f59e0b; padding:14px 18px;
             border-radius:10px; margin:18px 0; font-size:13.5px; line-height:1.5; }}
  .kpi-row {{ display:flex; gap:14px; flex-wrap:wrap; margin:18px 0; }}
  .kpi {{ flex:1 1 200px; background:white; padding:18px; border-radius:12px;
          box-shadow:0 2px 8px rgba(11,31,58,0.07); }}
  .kpi-value {{ font-size:28px; font-weight:700; color:{ACCENT}; }}
  .kpi-label {{ font-size:13px; color:{GREY}; margin-top:4px; }}
  .card {{ background:white; padding:22px; border-radius:12px;
           box-shadow:0 2px 8px rgba(11,31,58,0.07); margin:18px 0; }}
  .card h2 {{ margin:0 0 12px; font-size:18px; }}
  table.compare {{ width:100%; border-collapse:collapse; font-size:13.5px; }}
  table.compare th {{ text-align:left; background:{NAVY}; color:white;
                      padding:10px; font-weight:600; }}
  table.compare td {{ padding:9px 10px; border-bottom:1px solid #e5e7eb; }}
  table.compare tbody tr:nth-child(even) {{ background:#f9fafb; }}
  td.num {{ text-align:right; }}
  a {{ color:{ACCENT}; text-decoration:none; }}
  a:hover {{ text-decoration:underline; }}
  .narrative-headline {{ font-size:15px; font-weight:600; margin:0 0 14px; color:{NAVY}; }}
  .fact-grid {{ display:grid; grid-template-columns:repeat(2, 1fr); gap:10px 24px; margin-bottom:16px; }}
  .fact {{ border-left:3px solid {ACCENT}; padding-left:10px; }}
  .fact-label {{ font-size:11.5px; color:{GREY}; text-transform:uppercase; letter-spacing:0.04em; }}
  .fact-value {{ font-size:14.5px; font-weight:600; color:{NAVY}; }}
  .narrative-body {{ font-size:14px; line-height:1.6; margin:0; }}
  .open-btn {{ display:inline-block; background:{NAVY}; color:white; padding:12px 22px;
               border-radius:10px; text-decoration:none; font-weight:600; margin-top:8px; }}
  .open-btn:hover {{ background:{ACCENT}; }}
  footer {{ text-align:center; color:{GREY}; font-size:12px; margin:24px 0; }}
</style>
</head>
<body>
<div class="wrap">

  <header>
    <h1>ETF Cost Intelligence Engine</h1>
    <p>S&amp;P 500 UCITS ETF · Six Provider Comparison</p>
    <div class="generated">Generated {generated_at}</div>
  </header>

  <div class="banner">{DISCLAIMER_DASHBOARD}</div>

  {kpi_html}

  <div class="card">{chart_ter_html}</div>
  <div class="card">{chart_aum_html}</div>
  <div class="card">{chart_scatter_html}</div>

  <div class="card">
    <h2>Comparison table</h2>
    {table_html}
  </div>

  {narrative_card}

  <a class="open-btn" href="etf_dashboard.html" target="_blank">Open Dashboard in New Tab ↗</a>

  <footer>ETF Cost Intelligence Engine · analytical exercise, not investment advice.</footer>
</div>
</body>
</html>
"""

with open("etf_dashboard.html", "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print("=" * 78)
print("MODULE 5 COMPLETE — etf_dashboard.html written.")
print("=" * 78)
print(f"   File size: {len(dashboard_html):,} bytes")
print("   Source links embedded in comparison table.")
print("   Narrative rendered as fact grid plus body paragraph.")

MODULE 5 COMPLETE — etf_dashboard.html written.
   File size: 35,454 bytes
   Source links embedded in comparison table.
   Narrative rendered as fact grid plus body paragraph.


In [24]:
# ============================================================
# MODULE 6 — INLINE DISPLAY + OPEN-IN-NEW-TAB FALLBACK
# ============================================================
# 1) Render the dashboard inline via iframe (absolute path).
# 2) Provide a base64 data-URL button as a fallback that opens
#    the dashboard in a new tab even when local-file iframes
#    or in-page links are blocked (e.g. Colab).

import os

dashboard_path = os.path.abspath("etf_dashboard.html")

if not os.path.exists(dashboard_path):
    print("   WARNING: etf_dashboard.html not found. Run Module 5 first.")
else:
    # ---------- 6.1 INLINE IFRAME ----------
    display(IFrame(src=dashboard_path, width="100%", height=1400))